In [1]:
# 캐글 데이터셋 - 자전거 대여수요 예측

# ↓ 데이터 칼럼 보기

# 데이터 출처 : www.kaggle.com/c/bike/sharing-demand/data - 트레인셋과 테스트셋 csv 분리되있다
# datetime : 날짜/시간
# season : 1이면 봄, 2는 여름, 3은 가을, 4는 겨울
# holiday : 1은 토/일/휴일, 0은 휴일이 아닌 날
# workingday : 1은 주중, 0은 주말/휴일
# weather : 1은 맑음, 2는 안개, 3은 가벼운 눈/비, 4는 악천후(눈/비)
# temp : 온도
# atemp : 체감온도
# humidity : 습도
# windspeed : 풍속
# casual : 사전 등록하지 않은 대여횟수
# registered : 사전등록한 대여횟수
# count : 대여 횟수

# 데이터 클렌징 및 가공과 시각화
# 대여수 예측이라서 회귀모델 - 선형회귀, 릿지, 라쏘 테스트
# 랜덤포레스트, GBM, XGBoost, LigthGBM 모델 성능 테스트
# XGBoost나 라이트GBM은 설치되있나 확인

In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
df_train=pd.read_csv('bike_train.csv')
df_test=pd.read_csv('bike_test.csv')

In [3]:
df_test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 6493 entries, 0 to 6492
Data columns (total 9 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    6493 non-null   object 
 1   season      6493 non-null   int64  
 2   holiday     6493 non-null   int64  
 3   workingday  6493 non-null   int64  
 4   weather     6493 non-null   int64  
 5   temp        6493 non-null   float64
 6   atemp       6493 non-null   float64
 7   humidity    6493 non-null   int64  
 8   windspeed   6493 non-null   float64
dtypes: float64(3), int64(5), object(1)
memory usage: 456.7+ KB


In [4]:
df_train.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10886 entries, 0 to 10885
Data columns (total 12 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   datetime    10886 non-null  object 
 1   season      10886 non-null  int64  
 2   holiday     10886 non-null  int64  
 3   workingday  10886 non-null  int64  
 4   weather     10886 non-null  int64  
 5   temp        10886 non-null  float64
 6   atemp       10886 non-null  float64
 7   humidity    10886 non-null  int64  
 8   windspeed   10886 non-null  float64
 9   casual      10886 non-null  int64  
 10  registered  10886 non-null  int64  
 11  count       10886 non-null  int64  
dtypes: float64(3), int64(8), object(1)
memory usage: 1020.7+ KB


- 본격적으로 df_train=pd.read_csv('bike_train.csv') ← 이것을 가지고 미션 수행

In [5]:
# sk런 선택 1 - 선형회귀
from sklearn.linear_model import LinearRegression
lr=LinearRegression() # 모델 짓기

In [6]:
# datetime의 내용이 문자타입이라 특성공학을 써야 할 듯;;
df_train['datetime']=pd.to_datetime(df_train['datetime'])
df_train['dayofweek']=df_train['datetime'].dt.dayofweek
df_train['year']=df_train['datetime'].dt.year
df_train['month']=df_train['datetime'].dt.month
df_train['day']=df_train['datetime'].dt.day
df_train['hour']=df_train['datetime'].dt.hour

In [7]:
dayoption=df_train.drop(['datetime', 'casual', 'registered', 'count'], axis=1)
share_count=df_train['count']

In [8]:
X_train, X_test, y_train, y_test = train_test_split(dayoption, share_count, random_state=42)
lr.fit(X_train, y_train)

,fit_intercept,True
,copy_X,True
,tol,1e-06
,n_jobs,None
,positive,False


In [9]:
print(lr.score(X_train, y_train))
print(lr.score(X_test, y_test))

0.3897007136854159
0.38775297175889933


In [10]:
# sk런 선택 2 - 릿지
from sklearn.linear_model import Ridge
ridge=Ridge()
ridge.fit(X_train, y_train)
print(ridge.score(X_train, y_train))
print(ridge.score(X_test, y_test))

0.3897007004099623
0.3877513892697211


In [11]:
# sk런 선택 3 - 라쏘
from sklearn.linear_model import Lasso
lasso=Lasso()
lasso.fit(X_train, y_train)
print(lasso.score(X_train, y_train))
print(lasso.score(X_test, y_test))

0.38925134291213226
0.3872047013485169


In [12]:
# sk런 선택 4 - 랜덤 포레스트
from sklearn.ensemble import RandomForestRegressor
rf=RandomForestRegressor()
rf.fit(X_train, y_train)
print(rf.score(X_train, y_train))
print(rf.score(X_test, y_test))

0.9928296597517047
0.956028118801127


In [13]:
# sk런 선택 5 - 그라디언트 부스트
# 그라디언트 부스트는 y로 쓸 칼럼이 하나일 때만 가능
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.model_selection import cross_validate
gbr=GradientBoostingRegressor()
scores=cross_validate(gbr, X_train, y_train, return_train_score=True, n_jobs=-1)
gbr.fit(X_train, y_train)
print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))

0.8719993256092554
0.8620480032288962


- sk런 선택 6 - xgboost
- xgboost나 라이트GBM은 설치되있나 확인
- 기본적으로 안깔려있다 - !pip install xgboost

In [14]:
from xgboost import XGBRegressor

In [15]:
xgb=XGBRegressor(tree_method='hist', random_state=42)
scores=cross_validate(xgb, X_train, y_train, return_train_score=True, n_jobs=-1)
xgb.fit(X_train, y_train)
print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))

0.9882893204689026
0.9498883485794067


- sk런 테스트 7 - 라이트GBM
- 얘도 기본적으로 안깔려있다 - !pip install lightgbm

In [16]:
from lightgbm import LGBMRegressor
lgb=LGBMRegressor(random_state=42)
scores=cross_validate(lgb, X_train, y_train, return_train_score=True, n_jobs=-1)
lgb.fit(X_train, y_train)
print(np.mean(scores['train_score']))
print(np.mean(scores['test_score']))

[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000330 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 293
[LightGBM] [Info] Number of data points in the train set: 8164, number of used features: 13
[LightGBM] [Info] Start training from score 191.339784
0.9705264899088016
0.9501014431175857


- 모델별 회귀 평가

In [17]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, root_mean_squared_error

In [18]:
lr_y_pred_share_count=lr.predict(X_test)
ridge_y_pred_share_count=ridge.predict(X_test)
lasso_y_pred_share_count=lasso.predict(X_test)
rf_y_pred_share_count=rf.predict(X_test)
gbr_y_pred_share_count=gbr.predict(X_test)
xgb_y_pred_share_count=xgb.predict(X_test)
lgb_y_pred_share_count=lgb.predict(X_test)

In [19]:
# 회귀 성능평가 1 - 선형회귀
lr_mae=mean_absolute_error(y_test, lr_y_pred_share_count)
lr_mse=mean_squared_error(y_test, lr_y_pred_share_count)
lr_rmse=np.sqrt(lr_mse) # 루트 함수

print('---회귀 성능 평가---')
print(f'MAE(평균 절대오차) : {lr_mae:.2f}회')
print(f'MSE(평균 제곱오차) : {lr_mse:.2f}회')
print(f'RMSE1(제곱근 평균 제곱오차) : {lr_rmse:.2f}회')

---회귀 성능 평가---
MAE(평균 절대오차) : 105.50회
MSE(평균 제곱오차) : 20090.30회
RMSE1(제곱근 평균 제곱오차) : 141.74회


In [20]:
# 회귀 성능평가 2 - 릿지 회귀
ridge_mae=mean_absolute_error(y_test, ridge_y_pred_share_count)
ridge_mse=mean_squared_error(y_test, ridge_y_pred_share_count)
ridge_rmse=np.sqrt(ridge_mse) # 루트 함수

print('---회귀 성능 평가---')
print(f'MAE(평균 절대오차) : {ridge_mae:.2f}회')
print(f'MSE(평균 제곱오차) : {ridge_mse:.2f}회')
print(f'RMSE1(제곱근 평균 제곱오차) : {ridge_rmse:.2f}회')

---회귀 성능 평가---
MAE(평균 절대오차) : 105.50회
MSE(평균 제곱오차) : 20090.35회
RMSE1(제곱근 평균 제곱오차) : 141.74회


In [21]:
# 회귀 성능평가 3 - 라쏘 회귀
lasso_mae=mean_absolute_error(y_test, lasso_y_pred_share_count)
lasso_mse=mean_squared_error(y_test, lasso_y_pred_share_count)
lasso_rmse=np.sqrt(lasso_mse) # 루트 함수

print('---회귀 성능 평가---')
print(f'MAE(평균 절대오차) : {lasso_mae:.2f}회')
print(f'MSE(평균 제곱오차) : {lasso_mse:.2f}회')
print(f'RMSE1(제곱근 평균 제곱오차) : {lasso_rmse:.2f}회')

---회귀 성능 평가---
MAE(평균 절대오차) : 105.27회
MSE(평균 제곱오차) : 20108.29회
RMSE1(제곱근 평균 제곱오차) : 141.80회


In [22]:
# 회귀 성능평가 4 - 랜덤 포레스트
rf_mae=mean_absolute_error(y_test, rf_y_pred_share_count)
rf_mse=mean_squared_error(y_test, rf_y_pred_share_count)
rf_rmse=np.sqrt(rf_mse) # 루트 함수

print('---회귀 성능 평가---')
print(f'MAE(평균 절대오차) : {rf_mae:.2f}회')
print(f'MSE(평균 제곱오차) : {rf_mse:.2f}회')
print(f'RMSE1(제곱근 평균 제곱오차) : {rf_rmse:.2f}회')

---회귀 성능 평가---
MAE(평균 절대오차) : 24.02회
MSE(평균 제곱오차) : 1442.90회
RMSE1(제곱근 평균 제곱오차) : 37.99회


In [23]:
# 회귀 성능평가 5 - 그라디언트 부스트
gbr_mae=mean_absolute_error(y_test, gbr_y_pred_share_count)
gbr_mse=mean_squared_error(y_test, gbr_y_pred_share_count)
gbr_rmse=np.sqrt(gbr_mse) # 루트 함수

print('---회귀 성능 평가---')
print(f'MAE(평균 절대오차) : {gbr_mae:.2f}회')
print(f'MSE(평균 제곱오차) : {gbr_mse:.2f}회')
print(f'RMSE1(제곱근 평균 제곱오차) : {gbr_rmse:.2f}회')

---회귀 성능 평가---
MAE(평균 절대오차) : 44.86회
MSE(평균 제곱오차) : 4455.48회
RMSE1(제곱근 평균 제곱오차) : 66.75회


In [24]:
# 회귀 성능평가 6 - xg부스트
xgb_mae=mean_absolute_error(y_test, xgb_y_pred_share_count)
xgb_mse=mean_squared_error(y_test, xgb_y_pred_share_count)
xgb_rmse=np.sqrt(xgb_mse) # 루트 함수

print('---회귀 성능 평가---')
print(f'MAE(평균 절대오차) : {xgb_mae:.2f}회')
print(f'MSE(평균 제곱오차) : {xgb_mse:.2f}회')
print(f'RMSE1(제곱근 평균 제곱오차) : {xgb_rmse:.2f}회')

---회귀 성능 평가---
MAE(평균 절대오차) : 24.10회
MSE(평균 제곱오차) : 1421.83회
RMSE1(제곱근 평균 제곱오차) : 37.71회


In [25]:
# 회귀 성능평가 7 - 라이트gbm 회귀
lgb_mae=mean_absolute_error(y_test, lgb_y_pred_share_count)
lgb_mse=mean_squared_error(y_test, lgb_y_pred_share_count)
lgb_rmse=np.sqrt(lgb_mse) # 루트 함수

print('---회귀 성능 평가---')
print(f'MAE(평균 절대오차) : {lgb_mae:.2f}회')
print(f'MSE(평균 제곱오차) : {lgb_mse:.2f}회')
print(f'RMSE1(제곱근 평균 제곱오차) : {lgb_rmse:.2f}회')

---회귀 성능 평가---
MAE(평균 절대오차) : 24.13회
MSE(평균 제곱오차) : 1353.42회
RMSE1(제곱근 평균 제곱오차) : 36.79회
